# Case 4 – State Electricity Sell Price & NPV Analysis

**Setup:** Two staggered 1000 MW nuclear plants (combined model) + nat gas backup + fixed 2000 MW DC demand  
- Combined availability: 100% when both plants on, 50% when one is offline (staggered maintenance)
- Nat gas NGCT covers the 50% deficit during single-plant outages
- Nuclear surplus (when oversized) sold to grid via `grid_sell`

**Question:** At each US state's average retail electricity price, does this two-plant configuration produce a positive NPV?

**LCOE:** ~117.1 $/MWh at 2000 MW total (vs 97.4 for Case 3 always-on, premium from nat gas backup).

In [ ]:
import sys
sys.path.insert(0, '/Users/svijaysh/NPP+DCfork/H2Integrate')

import os
from h2integrate import EXAMPLE_DIR
os.chdir(EXAMPLE_DIR / '32_nuclear_DC_case_4')

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from h2integrate.core.h2integrate_model import H2IntegrateModel

warnings.filterwarnings('ignore')

DC_DEMAND_MW         = 2000
DC_DEMAND_KW         = DC_DEMAND_MW * 1e3
NUCLEAR_TOTAL_MW     = 2000          # 2 × 1000 MW
NUCLEAR_TOTAL_KW     = NUCLEAR_TOTAL_MW * 1e3
NG_FUEL_LARGE        = 1e8           # MMBtu/h — effectively unlimited

## 1. Staggered Availability Helpers

In [ ]:
def make_staggered_availability(
    n_hours=8760,
    maintenance_weeks=4,
    forced_outage_rate=0.001,
    plant1_maint_week=5,
    plant2_maint_week=31,
    seed=42,
):
    """Two per-plant availability arrays. Both plants never simultaneously offline."""
    np.random.seed(seed)
    maint_h = int(maintenance_weeks * 7 * 24)

    avail1 = np.ones(n_hours)
    m1s = (plant1_maint_week - 1) * 7 * 24
    avail1[m1s: m1s + maint_h] = 0.0

    avail2 = np.ones(n_hours)
    m2s = (plant2_maint_week - 1) * 7 * 24
    avail2[m2s: m2s + maint_h] = 0.0

    p1_on = np.where(avail1 == 1)[0]
    safe1 = p1_on[avail2[p1_on] == 1]
    n_f1  = min(int(len(p1_on) * forced_outage_rate), len(safe1))
    avail1[np.random.choice(safe1, size=n_f1, replace=False)] = 0.0

    p2_on = np.where(avail2 == 1)[0]
    safe2 = p2_on[avail1[p2_on] == 1]
    n_f2  = min(int(len(p2_on) * forced_outage_rate), len(safe2))
    avail2[np.random.choice(safe2, size=n_f2, replace=False)] = 0.0

    both_down = int(((avail1 == 0) & (avail2 == 0)).sum())
    assert both_down == 0, f'Both-down constraint violated: {both_down} h'
    print(f'  Plant 1: {avail1.mean()*100:.1f}% avail | {int((avail1==0).sum())} outage h/yr')
    print(f'  Plant 2: {avail2.mean()*100:.1f}% avail | {int((avail2==0).sum())} outage h/yr')
    print(f'  Simultaneous outage hours: {both_down} ✓')
    return avail1, avail2


def make_combined_avail(avail1, avail2):
    return (avail1 + avail2) / 2.0


def set_nuclear_outage_profile(h2i, nuclear_kw, combined_avail):
    h2i.prob.set_val('nuclear.electricity_demand', nuclear_kw * combined_avail, units='kW')


def set_natgas_fuel(h2i, fuel_mmbtu_per_h=NG_FUEL_LARGE):
    h2i.prob.set_val('nat_gas.natural_gas_in',
                     fuel_mmbtu_per_h * np.ones(8760), units='MMBtu/h')

## 2. State Electricity Price Data (EIA 2023)

In [ ]:
state_prices = {
    'AL': 12.41, 'AK': 22.54, 'AZ': 12.00, 'AR': 10.21, 'CA': 24.41,
    'CO': 12.29, 'CT': 25.77, 'DE': 13.36, 'DC': 15.02, 'FL': 13.09,
    'GA': 11.68, 'HI': 42.59, 'ID': 10.20, 'IL': 12.71, 'IN': 11.89,
    'IA':  9.95, 'KS': 11.56, 'KY': 10.55, 'LA': 10.33, 'ME': 22.43,
    'MD': 15.30, 'MA': 24.51, 'MI': 16.43, 'MN': 12.89, 'MS': 11.55,
    'MO': 11.02, 'MT': 10.88, 'NE':  9.98, 'NV': 12.11, 'NH': 23.41,
    'NJ': 17.18, 'NM': 12.44, 'NY': 21.77, 'NC': 11.79, 'ND':  9.71,
    'OH': 13.14, 'OK': 10.55, 'OR': 10.89, 'PA': 14.72, 'RI': 24.89,
    'SC': 12.07, 'SD': 11.08, 'TN': 11.59, 'TX': 12.01, 'UT': 10.15,
    'VT': 20.45, 'VA': 12.58, 'WA':  9.50, 'WV': 11.37, 'WI': 14.21,
    'WY':  8.97,
}

state_names = {
    'AL': 'Alabama',      'AK': 'Alaska',       'AZ': 'Arizona',      'AR': 'Arkansas',
    'CA': 'California',   'CO': 'Colorado',     'CT': 'Connecticut',  'DE': 'Delaware',
    'DC': 'D.C.',         'FL': 'Florida',      'GA': 'Georgia',      'HI': 'Hawaii',
    'ID': 'Idaho',        'IL': 'Illinois',     'IN': 'Indiana',      'IA': 'Iowa',
    'KS': 'Kansas',       'KY': 'Kentucky',     'LA': 'Louisiana',    'ME': 'Maine',
    'MD': 'Maryland',     'MA': 'Massachusetts','MI': 'Michigan',     'MN': 'Minnesota',
    'MS': 'Mississippi',  'MO': 'Missouri',     'MT': 'Montana',      'NE': 'Nebraska',
    'NV': 'Nevada',       'NH': 'New Hampshire','NJ': 'New Jersey',   'NM': 'New Mexico',
    'NY': 'New York',     'NC': 'North Carolina','ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma',     'OR': 'Oregon',       'PA': 'Pennsylvania', 'RI': 'Rhode Island',
    'SC': 'South Carolina','SD': 'South Dakota', 'TN': 'Tennessee',   'TX': 'Texas',
    'UT': 'Utah',         'VT': 'Vermont',      'VA': 'Virginia',     'WA': 'Washington',
    'WV': 'West Virginia','WI': 'Wisconsin',    'WY': 'Wyoming',
}

df_states = pd.DataFrame([
    {'state': k, 'name': state_names[k],
     'sell_price_¢/kWh': v,
     'sell_price_$/kWh': v / 100,
     'sell_price_$/MWh': v * 10}
    for k, v in state_prices.items()
]).sort_values('sell_price_$/MWh')

print(f"States: {len(df_states)}")
print(f"Price range: {df_states['sell_price_$/MWh'].min():.1f} – "
      f"{df_states['sell_price_$/MWh'].max():.1f} $/MWh")
print(f"Case 4 LCOE at {NUCLEAR_TOTAL_MW} MW total: ~117.1 $/MWh")

## 3. Initialize Model, Generate Availability Profile & Compute LCOE

In [ ]:
h2i = H2IntegrateModel('nuclear_datacenter_config_case_4.yaml')
h2i.setup()

print('Generating staggered availability (4-week maint + 0.1% forced outage each):')
avail1, avail2 = make_staggered_availability(maintenance_weeks=4, forced_outage_rate=0.001)
combined_avail = make_combined_avail(avail1, avail2)

h2i.prob.set_val('nuclear.system_capacity',        NUCLEAR_TOTAL_KW, units='kW')
h2i.prob.set_val('grid_sell.interconnection_size', NUCLEAR_TOTAL_KW, units='kW')
set_nuclear_outage_profile(h2i, NUCLEAR_TOTAL_KW, combined_avail)
set_natgas_fuel(h2i)

h2i.run()

lcoe_kwh = h2i.prob.get_val('finance_subgroup_nuclear.LCOE', units='USD/(kW*h)')[0]
lcoe_mwh = lcoe_kwh * 1000
nuc_cap  = h2i.prob.get_val('nuclear.CapEx',  units='USD')[0] / 1e9
ng_cap   = h2i.prob.get_val('nat_gas.CapEx',  units='USD')[0] / 1e6
ng_out   = h2i.prob.get_val('nat_gas.electricity_out', units='MW').flatten()

print(f'\nBaseline ({NUCLEAR_TOTAL_MW} MW total = 2 × {NUCLEAR_TOTAL_MW//2} MW):')
print(f'  Nuclear CapEx     : ${nuc_cap:.2f}B')
print(f'  Nat gas CapEx     : ${ng_cap:.0f}M')
print(f'  Nat gas run hours : {int((ng_out>1).sum())} h/yr')
print(f'  LCOE              : ${lcoe_mwh:.2f}/MWh')
print(f'  Break-even price  : {lcoe_mwh/10:.2f} ¢/kWh')

## 4. NPV Loop — All 51 States

The staggered availability profile is fixed (generated once). For each state's electricity price,  
the ProFAST NPV model is re-run with that sell price.

In [ ]:
results = []
n = len(df_states)

for i, row in df_states.iterrows():
    sp_kwh = row['sell_price_$/kWh']
    h2i.prob.set_val('finance_subgroup_nuclear_npv.sell_price_electricity',
                     sp_kwh, units='USD/(kW*h)')
    h2i.run()
    npv_b = h2i.prob.get_val('finance_subgroup_nuclear_npv.NPV_electricity', 'USD')[0] / 1e9
    results.append({
        'state':             row['state'],
        'name':              row['name'],
        'sell_price_¢/kWh': row['sell_price_¢/kWh'],
        'sell_price_$/MWh': row['sell_price_$/MWh'],
        'NPV_$B':            npv_b,
        'viable':            npv_b > 0,
    })
    idx = len(results)
    if idx % 10 == 0 or idx == n:
        print(f"  {idx}/{n}  {row['state']:2s}  "
              f"{row['sell_price_$/MWh']:6.1f} $/MWh  →  NPV {npv_b:+.2f} $B")

df_npv = pd.DataFrame(results)
viable_count = df_npv['viable'].sum()
print(f'\nViable states (NPV > 0): {viable_count} / {n}')
print(f'Break-even sell price  : {lcoe_mwh:.2f} $/MWh  ({lcoe_mwh/10:.2f} ¢/kWh)')

## 5. Results Table

In [ ]:
df_display = df_npv.sort_values('sell_price_$/MWh')[[
    'state','name','sell_price_¢/kWh','sell_price_$/MWh','NPV_$B','viable'
]].copy()
df_display['NPV_$B'] = df_display['NPV_$B'].round(3)
df_display['Status'] = df_display['viable'].map({True: '✓ Viable', False: '✗ Loss'})
print(df_display.to_string(index=False))

## 6. Bar Charts

In [ ]:
df_sorted = df_npv.sort_values('sell_price_$/MWh')
colors = ['#2ecc71' if v else '#e74c3c' for v in df_sorted['viable']]

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

ax = axes[0]
ax.bar(df_sorted['state'], df_sorted['sell_price_$/MWh'], color=colors, edgecolor='white', lw=0.5)
ax.axhline(lcoe_mwh, color='navy', lw=1.8, ls='--',
           label=f'LCOE = {lcoe_mwh:.1f} $/MWh (break-even)')
ax.set_ylabel('Electricity sell price ($/MWh)')
ax.set_title(f'Case 4 — State electricity prices vs LCOE '
             f'(2×{NUCLEAR_TOTAL_MW//2} MW nuclear + nat gas + {DC_DEMAND_MW} MW DC)')
ax.legend(fontsize=9)
ax.tick_params(axis='x', rotation=90, labelsize=7.5)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.bar(df_sorted['state'], df_sorted['NPV_$B'], color=colors, edgecolor='white', lw=0.5)
ax.axhline(0, color='black', lw=1)
ax.set_ylabel('NPV ($B)')
ax.set_title('NPV by State — green = profitable, red = loss')
ax.tick_params(axis='x', rotation=90, labelsize=7.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Viable: {viable_count}/51 states')

## 7. US Choropleth Maps

In [ ]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat>=4.2.0'],
               capture_output=True)
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [ ]:
fig = px.choropleth(
    df_npv,
    locations='state',
    locationmode='USA-states',
    color='sell_price_$/MWh',
    scope='usa',
    color_continuous_scale='RdYlGn',
    range_color=[df_npv['sell_price_$/MWh'].min(), df_npv['sell_price_$/MWh'].max()],
    labels={'sell_price_$/MWh': '$/MWh'},
    title='State Electricity Sell Prices (EIA 2023) — Case 4',
    hover_name='name',
    hover_data={'sell_price_¢/kWh': ':.2f', 'sell_price_$/MWh': ':.1f', 'state': False},
)
fig.update_layout(
    geo=dict(
        scope='usa',
        showlakes=True, lakecolor='#a8d8ea',
        showland=True,  landcolor='#f5f5f0',
        showcoastlines=True, coastlinecolor='gray',
    ),
    coloraxis_colorbar=dict(title='$/MWh', thickness=15),
    margin=dict(l=0, r=0, t=40, b=0),
    height=450,
)
fig.show()

In [ ]:
max_abs = df_npv['NPV_$B'].abs().max()

fig = px.choropleth(
    df_npv,
    locations='state',
    locationmode='USA-states',
    color='NPV_$B',
    scope='usa',
    color_continuous_scale='RdYlGn',
    range_color=[-max_abs, max_abs],
    labels={'NPV_$B': 'NPV ($B)'},
    title=(f'Case 4 — 30-yr NPV by State | 2×{NUCLEAR_TOTAL_MW//2} MW Nuclear + Nat Gas + '
           f'{DC_DEMAND_MW} MW DC | LCOE {lcoe_mwh:.1f} $/MWh'),
    hover_name='name',
    hover_data={'NPV_$B': ':.2f', 'sell_price_$/MWh': ':.1f', 'viable': True, 'state': False},
)
fig.update_layout(
    geo=dict(
        scope='usa',
        showlakes=True, lakecolor='#a8d8ea',
        showland=True,  landcolor='#f5f5f0',
        showcoastlines=True, coastlinecolor='gray',
    ),
    coloraxis_colorbar=dict(title='NPV ($B)', thickness=15),
    margin=dict(l=0, r=0, t=60, b=0),
    height=450,
)
fig.show()
print(f'Green = NPV > 0 (viable). Viable: {viable_count}/51 states.')

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'State Electricity Price ($/MWh)',
        f'30-yr NPV ($B) — {viable_count}/51 states viable',
    ],
    specs=[[{'type': 'choropleth'}, {'type': 'choropleth'}]],
)

geo_style = dict(
    scope='usa', showlakes=True, lakecolor='#a8d8ea',
    showland=True, landcolor='#f5f5f0',
    showcoastlines=True, coastlinecolor='gray',
)

fig.add_trace(
    go.Choropleth(
        locations=df_npv['state'], locationmode='USA-states',
        z=df_npv['sell_price_$/MWh'],
        colorscale='RdYlGn',
        colorbar=dict(title='$/MWh', x=0.46, thickness=12, len=0.8),
        hovertemplate='<b>%{text}</b><br>Price: %{z:.1f} $/MWh<extra></extra>',
        text=df_npv['name'],
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Choropleth(
        locations=df_npv['state'], locationmode='USA-states',
        z=df_npv['NPV_$B'],
        colorscale='RdYlGn',
        zmid=0,
        colorbar=dict(title='NPV ($B)', x=1.0, thickness=12, len=0.8),
        hovertemplate='<b>%{text}</b><br>NPV: %{z:.2f} $B<extra></extra>',
        text=df_npv['name'],
    ),
    row=1, col=2,
)

fig.update_geos(geo_style)
fig.update_layout(
    title_text=(f'Case 4: 2×{NUCLEAR_TOTAL_MW//2} MW Nuclear + Nat Gas + {DC_DEMAND_MW} MW DC | '
                f'LCOE {lcoe_mwh:.1f} $/MWh | Break-even {lcoe_mwh/10:.2f} ¢/kWh'),
    height=420, margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()

## 8. Regional Summary

In [ ]:
region_map = {
    'New England':    ['CT','ME','MA','NH','RI','VT'],
    'Mid-Atlantic':   ['DC','DE','MD','NJ','NY','PA'],
    'South':          ['AL','AR','FL','GA','KY','LA','MS','NC','SC','TN','VA','WV'],
    'Midwest':        ['IL','IN','IA','KS','MI','MN','MO','NE','ND','OH','SD','WI'],
    'Mountain':       ['AZ','CO','ID','MT','NV','NM','UT','WY'],
    'Pacific':        ['AK','CA','HI','OR','WA'],
    'Texas':          ['TX'],
    'Oklahoma':       ['OK'],
}
df_npv['region'] = df_npv['state'].map(
    {s: r for r, states in region_map.items() for s in states}
).fillna('Other')

regional = df_npv.groupby('region').agg(
    avg_price=('sell_price_$/MWh','mean'),
    avg_npv=('NPV_$B','mean'),
    viable_states=('viable','sum'),
    total_states=('viable','count'),
).reset_index()
regional['viable_pct'] = (regional['viable_states'] / regional['total_states'] * 100).round(1)
regional = regional.sort_values('avg_price', ascending=False)
print(regional.to_string(index=False, float_format='%.1f'))

fig, ax = plt.subplots(figsize=(10, 5))
colors_r = ['#2ecc71' if p >= lcoe_mwh else '#e74c3c' for p in regional['avg_price']]
ax.bar(regional['region'], regional['avg_price'], color=colors_r, edgecolor='white')
ax.axhline(lcoe_mwh, color='navy', ls='--', lw=1.8, label=f'LCOE {lcoe_mwh:.1f} $/MWh')
ax.set_ylabel('Avg sell price ($/MWh)')
ax.set_title('Case 4 — Average Electricity Price by Region')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. NPV vs Price Scatter

In [ ]:
fig = px.scatter(
    df_npv,
    x='sell_price_$/MWh', y='NPV_$B',
    color='viable',
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    text='state',
    title=(f'Case 4: NPV vs Electricity Sell Price '
           f'(2×{NUCLEAR_TOTAL_MW//2} MW nuclear + nat gas + {DC_DEMAND_MW} MW DC)'),
    labels={'sell_price_$/MWh': 'State sell price ($/MWh)', 'NPV_$B': 'NPV ($B)'},
    hover_name='name',
)
fig.add_vline(x=lcoe_mwh, line_dash='dash', line_color='navy',
              annotation_text=f'LCOE = {lcoe_mwh:.1f} $/MWh', annotation_position='top right')
fig.add_hline(y=0, line_width=1, line_color='black')
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 10. Summary

In [ ]:
best = df_npv.nlargest(5, 'NPV_$B')[['state','name','sell_price_$/MWh','NPV_$B']]
worst = df_npv.nsmallest(5, 'NPV_$B')[['state','name','sell_price_$/MWh','NPV_$B']]

print('=' * 65)
print(f'Case 4 — 2×{NUCLEAR_TOTAL_MW//2} MW Staggered Nuclear + Nat Gas + {DC_DEMAND_MW} MW DC')
print(f'  LCOE               : {lcoe_mwh:.2f} $/MWh')
print(f'  Break-even price   : {lcoe_mwh/10:.2f} ¢/kWh')
print(f'  Viable states      : {viable_count} / 51')
print(f'  NPV range          : {df_npv["NPV_$B"].min():.2f} to {df_npv["NPV_$B"].max():.2f} $B')
print()
print('Top 5 states (best NPV):')
print(best.to_string(index=False, float_format='%.2f'))
print()
print('Bottom 5 states (worst NPV):')
print(worst.to_string(index=False, float_format='%.2f'))
print('=' * 65)